In [11]:
import faiss
from sentence_transformers import SentenceTransformer
import os

model = SentenceTransformer("all-MiniLM-L6-v2")


In [12]:
# load text from a file
texts = None
with open('english-5k.csv', 'r', encoding='utf-8') as f:
    texts = [line.strip() for line in f]


In [13]:
vector_db_path = "english_5k_hnsw.index"


In [14]:
if not os.path.exists(vector_db_path): # skip creating the db if it already exists
    embeddings = model.encode(texts, convert_to_numpy=True)
    embeddings = embeddings.astype("float32")
    print("embeddings.shape:", embeddings.shape)

    d = embeddings.shape[1]
    M = 32
    index = faiss.IndexHNSWFlat(d, M)
    index.add(embeddings)
    print("Total vectors in index:", index.ntotal)

    # save the index
    faiss.write_index(index, vector_db_path)


In [15]:
# load the index
index = faiss.read_index(vector_db_path)


In [ ]:
query = "king"
query_embedding = model.encode([query], convert_to_numpy=True)
query_embedding = query_embedding.astype("float32")

k = 5  # number of nearest neighbors to retrieve
D, I = index.search(query_embedding, k)
print(f"Top {k} nearest neighbors for '{query}':")
for dist, idx in zip(D[0], I[0]):
    print(f"Word: {texts[idx]}, Index: {idx}, Distance: {dist}")

# try to get queen with adding vectors together
q1 = "king"
q2 = "male"
q3 = "female"

q1_emb = model.encode([q1], convert_to_numpy=True).astype("float32")
q2_emb = model.encode([q2], convert_to_numpy=True).astype("float32")
q3_emb = model.encode([q3], convert_to_numpy=True).astype("float32")

combined_emb = q1_emb - q2_emb + q3_emb
D, I = index.search(combined_emb, k)
print(f"\nTop {k} nearest neighbors for combination of '{q1}' - '{q2}' + '{q3}':")
for dist, idx in zip(D[0], I[0]):
    print(f"Word: {texts[idx]}, Index: {idx}, Distance: {dist}")


Top 5 nearest neighbors for 'king':
Word: king, Index: 2082, Distance: 9.338125975602574e-13
Word: kingdom, Index: 3948, Distance: 0.5331282615661621
Word: queen, Index: 3616, Distance: 0.6385747194290161
Word: royal, Index: 3786, Distance: 0.7182959318161011
Word: champion, Index: 2441, Distance: 0.860819935798645

Top 5 nearest neighbors for combination of 'king' - 'male' + 'female':
Word: king, Index: 2082, Distance: 0.532413899898529
Word: queen, Index: 3616, Distance: 0.7292802333831787
Word: kingdom, Index: 3948, Distance: 0.9252558350563049
Word: royal, Index: 3786, Distance: 1.064408779144287
Word: champion, Index: 2441, Distance: 1.196191668510437
